# EDA: Olist Brazilian E-Commerce Dataset

สำรวจโครงสร้างข้อมูล (`.info()`), ค่า null, ค่าซ้ำ, และ unique values เฉพาะ 4 ไฟล์ที่เราจะ clean:
`olist_customers_dataset.csv`, `olist_geolocation_dataset.csv`, `olist_products_dataset.csv`, `olist_sellers_dataset.csv`

(ไม่รวมไฟล์กลุ่ม order และ `product_category_name_translation.csv`)

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

TARGET_FILES = [
    'olist_customers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
]
files = [f'archive/{f}' for f in TARGET_FILES]
files

In [ ]:
# โหลดทุกไฟล์เก็บไว้ใน dict เพื่อเรียกใช้ทีหลัง
dfs = {f.split('\\')[-1].split('/')[-1]: pd.read_csv(f) for f in files}
list(dfs.keys())

## 1. Info + Null + Duplicate rows ของทั้ง 4 ไฟล์

In [ ]:
for name, df in dfs.items():
    print('=' * 100)
    print(name)
    print('=' * 100)
    print(f"Shape: {df.shape}")
    print()
    df.info()
    print()
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print("Null counts (columns with nulls only):")
    print(nulls if len(nulls) > 0 else "  (none)")
    print()
    print("Duplicate rows:", df.duplicated().sum())
    print()

## 2. Unique values ต่อคอลัมน์ (หา Primary Key candidates)

In [ ]:
for name, df in dfs.items():
    print('=' * 100)
    print(name)
    print('=' * 100)
    n = len(df)
    print(f"Total rows: {n}")
    for col in df.columns:
        nun = df[col].nunique()
        flag = " <- ALL UNIQUE (PK candidate)" if nun == n else (" <- SINGLE VALUE" if nun == 1 else "")
        print(f"  {col:35s} nunique={nun:<10d} ratio={nun/n:.3f}{flag}")
    print()

## 3. ดูตารางแยกทีละไฟล์ (แก้ชื่อไฟล์ตรงนี้เพื่อดูตารางที่ต้องการ)

In [ ]:
name = 'olist_customers_dataset.csv'
df = dfs[name]
df.head()

In [ ]:
df.describe(include='all').T

## 4. ตัวอย่าง: ตรวจสอบ customer_id vs customer_unique_id

In [ ]:
customers = dfs['olist_customers_dataset.csv']
print("customer_id unique:", customers['customer_id'].nunique())
print("customer_unique_id unique:", customers['customer_unique_id'].nunique())

# ลูกค้าที่สั่งซื้อมากกว่า 1 ครั้ง (มี customer_id หลายอันแต่ customer_unique_id เดียวกัน)
repeat_customers = (
    customers.groupby('customer_unique_id')['customer_id']
    .nunique()
    .sort_values(ascending=False)
)
repeat_customers[repeat_customers > 1].head(10)

## 5. ตัวอย่าง: ตรวจสอบแถวซ้ำใน geolocation

In [ ]:
geo = dfs['olist_geolocation_dataset.csv']
print("Duplicate rows:", geo.duplicated().sum())
geo[geo.duplicated(keep=False)].sort_values('geolocation_zip_code_prefix').head(10)

# 6. Data Cleaning

ทำความสะอาดทั้ง 4 ไฟล์ในสโคปนี้:
- `olist_customers_dataset.csv`
- `olist_geolocation_dataset.csv`
- `olist_products_dataset.csv`
- `olist_sellers_dataset.csv`

หลักการ: แก้เฉพาะสิ่งที่เป็น **data quality bug ที่ยืนยันแล้วจาก section 1-5** ไม่เดาเพิ่มเอง และจะ**ไม่ทิ้งข้อมูล** เว้นแต่จำเป็นจริงๆ — กรณีที่ยังไม่แน่ใจ (duplicate rows และ outlier พิกัดใน geolocation) จะ**เก็บข้อมูลไว้ครบ แต่แปะ flag** ให้ตรวจสอบภายหลังแทนการลบทิ้งเลย

In [ ]:
customers = dfs['olist_customers_dataset.csv']
geo = dfs['olist_geolocation_dataset.csv']
products = dfs['olist_products_dataset.csv']
sellers = dfs['olist_sellers_dataset.csv']

## 6.1 Customers — แก้ zip code prefix

**ปัญหา:** `customer_zip_code_prefix` ถูกอ่านเป็น `int64` ทำให้เลข 0 นำหน้าหายไป (เช่น `"09790"` กลายเป็น `9790`) กระทบ 23,995 จาก 99,441 แถว (~24%)

**ทำไมต้องแก้:** zip code prefix ไม่ใช่ตัวเลขเชิงปริมาณ (จะเอาไปบวก/ลบ/เฉลี่ยไม่ได้อยู่แล้ว) การเก็บเป็น int จึงผิดประเภทข้อมูลตั้งแต่ต้น และถ้าปล่อยไว้จะ join กับ `geolocation`/`sellers` (ที่ต้องแก้ zip แบบเดียวกัน) ผิดคู่แบบเงียบๆ

**วิธีแก้:** แปลงเป็น `str` แล้ว `zfill(5)` เติม 0 นำหน้าคืนให้ครบ 5 หลักตามมาตรฐาน zip ของบราซิล

In [ ]:
def clean_customers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['customer_zip_code_prefix'] = df['customer_zip_code_prefix'].astype(str).str.zfill(5)
    assert df['customer_zip_code_prefix'].str.len().eq(5).all()
    return df


customers_clean = clean_customers(customers)
customers_clean.head()

## 6.2 Geolocation — แก้ zip, trim ช่องว่าง, ลบ outlier พิกัด, **flag** (ไม่ลบ) duplicate

**ปัญหาที่แก้ตรงๆ:**
1. `geolocation_zip_code_prefix` เป็น `int64` เจอปัญหาเดียวกับ customers → แปลงเป็น `str.zfill(5)`
2. `geolocation_city` มี 1 แถวที่มีช่องว่างหน้า/หลังติดมา → `.str.strip()`

**พิกัดผิดปกติ — ตัดสินใจแล้วว่า "ลบ" (อัปเดตจากเดิมที่ flag ไว้เฉยๆ):**
เช็ค 2 เกณฑ์: (1) หลุด bounding box บราซิลทั้งประเทศ (lat -34..6, lng -74..-32) (2) ห่างจากพิกัด **median ของ zip prefix เดียวกัน** (haversine distance) เกิน **100 กม.** — เพราะพิกัดที่ "อยู่ในประเทศ" ก็ยังผิดเมืองไปไกลได้ ซึ่ง bbox ทั้งประเทศจับไม่ได้

- ทำไมใช้ 100 กม.: จาก distribution จริงพบว่า 75% ของแถวห่างจาก median ไม่ถึง 1 กม. (zip prefix ปกติควรกระจุกตัวแคบๆ) แต่ tail ยาวไปถึง ~9,500 กม. เลือก 100 กม. เพื่อไม่ไป flag พื้นที่ชนบทที่ zip เดียวกันครอบคลุมพื้นที่กว้างตามธรรมชาติ แต่ยังจับ error ชัดเจนได้
- ทำไมใช้ **median** ของ zip แทน mean: median ทนต่อ outlier ในกลุ่มเดียวกันมากกว่า
- รวม 2 เกณฑ์เป็น `is_coord_outlier` แล้ว**ลบแถวที่เป็น True ออกจริง** (840 แถว) เพราะตรวจสอบแล้วมั่นใจว่าเป็น geocoding error ไม่ใช่ข้อมูลที่ถูกต้องแค่หายาก — ต่างจาก duplicate ที่ยังไม่มีหลักฐานพอจะฟันธง
- หลังลบแล้ว **ตัดคอลัมน์ช่วยเช็คทิ้งทั้งหมด** (`dist_from_zip_median_km`, `is_coord_outlier`, `is_duplicate_row`) เพราะใช้เสร็จภารกิจแล้ว ไม่ต้องพกติดไปกับข้อมูลที่ใช้งานจริงต่อ

**Duplicate rows เต็มแถว (26% ของข้อมูลเดิม):** ยังคง**ไม่ลบ**เหมือนเดิม — อาจจะจริง (คนละคนอยู่ zip เดียวกัน พิกัดเดียวกันได้) หรืออาจเป็นข้อมูลซ้ำจาก source เดิม ยังไม่มีหลักฐานเหมือนกรณี outlier ที่เช็คระยะทางแล้วฟันธงได้ชัดเจน จึงยังไม่ dedupe รอบนี้

In [ ]:
GEO_LAT_MIN, GEO_LAT_MAX = -34, 6
GEO_LNG_MIN, GEO_LNG_MAX = -74, -32
GEO_DIST_THRESHOLD_KM = 100


def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def clean_geolocation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['geolocation_zip_code_prefix'] = df['geolocation_zip_code_prefix'].astype(str).str.zfill(5)
    df['geolocation_city'] = df['geolocation_city'].str.strip()

    # เกณฑ์ที่ 1: พิกัดหลุดขอบเขตบราซิลทั้งประเทศ
    out_of_country = (
        (df['geolocation_lat'] > GEO_LAT_MAX) | (df['geolocation_lat'] < GEO_LAT_MIN) |
        (df['geolocation_lng'] > GEO_LNG_MAX) | (df['geolocation_lng'] < GEO_LNG_MIN)
    )

    # เกณฑ์ที่ 2: พิกัดห่างจาก median ของ zip prefix เดียวกันเกิน threshold
    zip_median = (
        df.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']]
        .median()
        .rename(columns={'geolocation_lat': 'zip_median_lat', 'geolocation_lng': 'zip_median_lng'})
    )
    df = df.merge(zip_median, on='geolocation_zip_code_prefix', how='left')
    df['dist_from_zip_median_km'] = haversine_km(
        df['geolocation_lat'], df['geolocation_lng'],
        df['zip_median_lat'], df['zip_median_lng'],
    )
    far_from_zip_cluster = df['dist_from_zip_median_km'] > GEO_DIST_THRESHOLD_KM

    df['is_coord_outlier'] = out_of_country | far_from_zip_cluster
    df = df.drop(columns=['zip_median_lat', 'zip_median_lng'])

    check_cols = [c for c in df.columns if c not in ('is_coord_outlier', 'dist_from_zip_median_km')]
    df['is_duplicate_row'] = df.duplicated(subset=check_cols, keep=False)

    # ลบแถว outlier ออกจริง (ยืนยันแล้วว่าเป็น geocoding error) แล้วตัดคอลัมน์ช่วยเช็คทิ้ง
    # ส่วน duplicate rows ยังไม่ลบ (ตามที่ตกลงกันไว้ก่อนหน้า)
    n_before = len(df)
    df = df[~df['is_coord_outlier']].copy()
    print(f"ลบ outlier ออก {n_before - len(df)} แถว (เหลือ {len(df)} แถว)")
    df = df.drop(columns=['dist_from_zip_median_km', 'is_coord_outlier', 'is_duplicate_row'])
    return df


geo_clean = clean_geolocation(geo)
geo_clean.head()

### Scatter plot: จุดปกติ vs จุด outlier ที่ห่างจาก median ของ zip ตัวเอง

พล็อตพิกัดทุกจุดเป็น longitude (x) / latitude (y) เพื่อดูภาพรวมว่า outlier ที่เช็คด้วยระยะห่างจาก median ของ zip (840 แถวที่ลบไปแล้วใน `clean_geolocation`) กระจายอยู่ตรงไหนบ้างเทียบกับรูปร่างของบราซิลจริง — จุดสีน้ำเงินคือข้อมูลปกติ (สุ่มมาโชว์ 50,000 จุดเพราะข้อมูลมี ~1 ล้านแถว plot หมดจะช้าและทับกันจนดูไม่ออก) จุดสีส้มคือ outlier ทั้งหมด (840 จุด แสดงครบทุกจุดเพราะจำนวนน้อย)

สังเกตได้ว่า outlier บางจุดหลุดออกนอกทวีปอเมริกาใต้ไปเลย (ยุโรป/แอฟริกา/เอเชีย) ส่วนอีกจำนวนมากอยู่ปะปนอยู่ในรูปร่างบราซิลเอง — เป็นเคสที่พิกัด "ดูปกติในสายตา" แต่จริงๆ ห่างจากพิกัดอื่นๆ ใน zip เดียวกันเกิน 100 กม. ซึ่งกราฟนี้ช่วยยืนยันว่าเกณฑ์ที่ 2 (ระยะห่างจาก zip median) จำเป็นจริง ไม่ใช่แค่เช็ค bounding box อย่างเดียวพอ

In [ ]:
import matplotlib.pyplot as plt

COLOR_NORMAL = '#2a78d6'   # categorical slot 1
COLOR_OUTLIER = '#eb6834'  # categorical slot 2

# คำนวณ flag ใหม่จาก geo ดิบ (เหมือนใน clean_geolocation) เพราะ geo_clean ตัดคอลัมน์ flag ทิ้งไปแล้ว
_zip = geo.copy()
_zip['geolocation_zip_code_prefix'] = _zip['geolocation_zip_code_prefix'].astype(str).str.zfill(5)
_out_of_country = (
    (_zip['geolocation_lat'] > GEO_LAT_MAX) | (_zip['geolocation_lat'] < GEO_LAT_MIN) |
    (_zip['geolocation_lng'] > GEO_LNG_MAX) | (_zip['geolocation_lng'] < GEO_LNG_MIN)
)
_zip_median = (
    _zip.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']]
    .median()
    .rename(columns={'geolocation_lat': 'zip_median_lat', 'geolocation_lng': 'zip_median_lng'})
)
_zip = _zip.merge(_zip_median, on='geolocation_zip_code_prefix', how='left')
_zip['dist_from_zip_median_km'] = haversine_km(
    _zip['geolocation_lat'], _zip['geolocation_lng'], _zip['zip_median_lat'], _zip['zip_median_lng']
)
_zip['is_coord_outlier'] = _out_of_country | (_zip['dist_from_zip_median_km'] > GEO_DIST_THRESHOLD_KM)

normal_points = _zip[~_zip['is_coord_outlier']]
outlier_points = _zip[_zip['is_coord_outlier']]
sample_normal = normal_points.sample(50_000, random_state=42)

fig, ax = plt.subplots(figsize=(9, 9))
ax.set_facecolor('#fcfcfb')
fig.patch.set_facecolor('#fcfcfb')

ax.scatter(sample_normal['geolocation_lng'], sample_normal['geolocation_lat'],
           s=3, c=COLOR_NORMAL, alpha=0.25, linewidths=0,
           label=f'Normal (sample {len(sample_normal):,} of {len(normal_points):,} rows)')
ax.scatter(outlier_points['geolocation_lng'], outlier_points['geolocation_lat'],
           s=35, c=COLOR_OUTLIER, alpha=0.9, edgecolors='white', linewidths=0.6,
           label=f'Outlier ({len(outlier_points)} rows, >100km from own zip median)')

ax.set_xlabel('Longitude', color='#52514e')
ax.set_ylabel('Latitude', color='#52514e')
ax.set_title('Geolocation coordinates: normal vs distance-from-zip-median outliers', color='#0b0b0b')
ax.tick_params(colors='#898781')
ax.grid(True, color='#e1e0d9', linewidth=0.6)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_color('#c3c2b7')
ax.legend(loc='lower left', frameon=False, labelcolor='#0b0b0b')

plt.tight_layout()
plt.show()

## 6.3 Products — เปลี่ยนชื่อคอลัมน์พิมพ์ผิด + จัดการ null

**1) แก้ชื่อคอลัมน์สะกดผิดในไฟล์ต้นฉบับ:** `product_name_lenght` → `product_name_length`, `product_description_lenght` → `product_description_length`

ทำไม: คำว่า "lenght" เป็นการสะกดผิด (ที่ถูกคือ "length") ถ้าไม่แก้จะทำให้เขียนโค้ดผิดซ้ำๆ ต่อไปทั้งกลุ่ม และเวลาอ้างอิงในรายงาน/โค้ดอื่นจะดูไม่เป็นมืออาชีพ ไม่กระทบข้อมูล กระทบแค่ชื่อคอลัมน์

**2) null ที่มาเป็นชุดเดียวกัน 610 แถว** (`product_category_name`, `product_name_length`, `product_description_length`, `product_photos_qty` เป็น null พร้อมกัน):
- `product_category_name` → fill เป็น `"unknown"` แทนการลบทิ้ง เพราะ 610/32,951 (1.8%) เป็นสัดส่วนน้อย และคอลัมน์อื่นของแถวเหล่านี้ (เช่น weight/dimensions) ส่วนใหญ่ยังมีข้อมูลใช้ได้ปกติ การลบทิ้งจะเสีย record ที่ยังมีประโยชน์ไปโดยไม่จำเป็น
- `product_name_length`, `product_description_length`, `product_photos_qty` → fill เป็น `0` เพราะ 3 คอลัมน์นี้นับจำนวน/ความยาว ค่า `0` สื่อความหมายตรงไปตรงมาว่า "ไม่มีข้อมูลชื่อ/คำอธิบาย/รูปสินค้า" ต่างจากค่าจริงที่จะมากกว่า 0 เสมอ จึงไม่ทำให้ปนกับข้อมูลจริงโดยไม่ตั้งใจ

**⚠️ หมายเหตุที่แปลกและยังไม่ได้ข้อสรุป (ตามที่ขอให้บันทึกไว้):** ทำไม category, name length, description length, photos qty ถึง null พร้อมกันเป๊ะทั้ง 610 แถว แต่ราคา/น้ำหนัก/ขนาดกลับมีข้อมูลปกติ — ลักษณะนี้ดูเหมือนสินค้าที่ import เข้าระบบแบบ bulk โดยไม่ได้กรอก listing (ชื่อ/คำอธิบาย/รูป/หมวดหมู่) แต่มีข้อมูล logistics (น้ำหนัก/ขนาด) มาจาก supplier อยู่แล้ว เป็นสมมติฐาน ยังไม่ยืนยัน ควรเช็คกับต้นทาง Olist หรือดูว่าสินค้ากลุ่มนี้ผูกกับ order/seller อะไรเป็นพิเศษหรือไม่

**3) อีก 2 แถวที่ไม่มีข้อมูล weight/length/height/width เลย** → fill ด้วย **median** ของแต่ละคอลัมน์ เพราะมีแค่ 2 จาก 32,951 แถว (0.006%) ผลกระทบจากการ impute แทบไม่มีนัยสำคัญ และ median ทนทานต่อ outlier มากกว่า mean

In [ ]:
def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={
        'product_name_lenght': 'product_name_length',
        'product_description_lenght': 'product_description_length',
    }).copy()

    df['product_category_name'] = df['product_category_name'].fillna('unknown')

    for col in ['product_name_length', 'product_description_length', 'product_photos_qty']:
        df[col] = df[col].fillna(0).astype(int)

    for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
        df[col] = df[col].fillna(df[col].median())

    assert df.isnull().sum().sum() == 0
    return df


products_clean = clean_products(products)
products_clean.head()

## 6.4 Sellers — แก้ zip code prefix

ปัญหาและวิธีแก้เหมือน customers/geolocation ทุกประการ: `seller_zip_code_prefix` เป็น `int64` ทำให้เลข 0 นำหน้าหาย ต้องแปลงเป็น `str.zfill(5)` เพื่อให้ zip ยาว 5 หลักเสมอและ join กับตารางอื่นได้ถูกต้อง ไม่มี null/duplicate/whitespace ให้แก้เพิ่ม (เช็คแล้วจาก section 1)

In [ ]:
def clean_sellers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['seller_zip_code_prefix'] = df['seller_zip_code_prefix'].astype(str).str.zfill(5)
    assert df['seller_zip_code_prefix'].str.len().eq(5).all()
    return df


sellers_clean = clean_sellers(sellers)
sellers_clean.head()

## 6.5 บันทึกไฟล์ที่ clean แล้ว

เซฟลงโฟลเดอร์ `archive/clean/` แยกจากไฟล์ดิบเดิม เพื่อไม่ให้ทับไฟล์ต้นฉบับ และให้ขั้นตอนถัดไป (feature engineering / join กับ order) เรียกใช้ไฟล์ clean ได้ตรงๆ โดยไม่ต้องรันโค้ด clean ซ้ำทุกครั้ง

ไฟล์ `orders_*` และ `product_category_name_translation.csv` **ไม่ได้ clean ในรอบนี้** ตามที่ตกลงกันไว้ — ยังคงอยู่ใน `archive/` เดิม

In [ ]:
import os

out_dir = 'archive/clean'
os.makedirs(out_dir, exist_ok=True)

customers_clean.to_csv(f'{out_dir}/olist_customers_dataset_clean.csv', index=False)
geo_clean.to_csv(f'{out_dir}/olist_geolocation_dataset_clean.csv', index=False)
products_clean.to_csv(f'{out_dir}/olist_products_dataset_clean.csv', index=False)
sellers_clean.to_csv(f'{out_dir}/olist_sellers_dataset_clean.csv', index=False)

print("บันทึกไฟล์ clean แล้วที่:", os.path.abspath(out_dir))
for f in sorted(os.listdir(out_dir)):
    print(" -", f)